In [1]:
import torch
from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from dengue_envs.envs.dengue_diagnostics import DengueDiagnosticsEnv
from dengue_wrapper import DengueWrapper, CaseByCaseWrapper
from fcn_network import DengueNet

C:\Users\segun\AppData\Local\pypoetry\Cache\virtualenvs\dengue-diagnostics-env-Ra5buD89-py3.12\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


In [2]:
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: False


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LR = 1e-4
GAMMA = 0.99
N_STEP = 3
TARGET_UPDATE_FREQ = 1000

BUFFER_SIZE = 1000
BATCH_SIZE = 64

EPOCH = 10
STEP_PER_EPOCH = 10000

STEP_PER_COLLECT = 1000
UPDATE_PER_STEP = 0.1

EPS_TRAIN_START = 1.0
EPS_TRAIN_FINAL = 0.05
EPS_TRAIN_DECAY = 50000
EPS_TEST = 0.01
NUM_ENVS = 4
NUM_TEST_ENVS = 4

In [4]:
def make_env():
    """Função factory para criar o ambiente com wrappers."""
    env = DengueDiagnosticsEnv(epilength=60, size=400, clinical_specificity=(0.5, 0.95))
    env = DengueWrapper(env)
    env = CaseByCaseWrapper(env)
    return env

In [5]:
print(f"VERIFICAÇÃO: O BUFFER_SIZE é {BUFFER_SIZE}")

VERIFICAÇÃO: O BUFFER_SIZE é 1000


In [6]:
if True:
    train_envs = DummyVectorEnv([make_env for _ in range(NUM_ENVS)])
    test_envs = DummyVectorEnv([make_env for _ in range(NUM_TEST_ENVS)])

    env = make_env()
    map_shape = env.observation_space.spaces["map"].shape
    action_shape = env.action_space.n

    net = DengueNet(map_shape, action_shape, device=DEVICE).to(DEVICE)
    optim = torch.optim.Adam(net.parameters(), lr=LR)

    policy = DQNPolicy(
        model=net,
        optim=optim,
        discount_factor=GAMMA,
        estimation_step=N_STEP,
        target_update_freq=TARGET_UPDATE_FREQ,
        action_space=env.action_space
    )

    buffer = VectorReplayBuffer(
        total_size=BUFFER_SIZE,
        buffer_num=NUM_ENVS,
        ignore_obs_next=True
    )

    train_collector = Collector(
        policy, train_envs, buffer, exploration_noise=True
    )
    test_collector = Collector(policy, test_envs)


    def train_fn(epoch, env_step):
        if env_step <= EPS_TRAIN_DECAY:
            eps = EPS_TRAIN_START - env_step / EPS_TRAIN_DECAY * \
                  (EPS_TRAIN_START - EPS_TRAIN_FINAL)
        else:
            eps = EPS_TRAIN_FINAL
        policy.set_eps(eps)


    def test_fn(epoch, env_step):
        policy.set_eps(EPS_TEST)

    trainer = OffpolicyTrainer(
        policy=policy,
        train_collector=train_collector,
        test_collector=test_collector,
        max_epoch=EPOCH,
        step_per_epoch=STEP_PER_EPOCH,
        step_per_collect=STEP_PER_COLLECT,
        update_per_step=UPDATE_PER_STEP,
        episode_per_test=NUM_TEST_ENVS,
        batch_size=BATCH_SIZE,
        train_fn=train_fn,
        test_fn=test_fn,
        stop_fn=lambda mean_rewards: mean_rewards >= 100
    )

    print(f"Iniciando treinamento na {DEVICE}...")
    result = trainer.run()
    print("\n--- Resultado do Treinamento ---")
    print(result)

    torch.save(policy.state_dict(), "dqn_dengue_policy2.pth")
    print("Política salva em dqn_dengue_policy.pth")

Iniciando treinamento na cpu...
Reward: -1.0 	 Total Reward: -1.0
Reward: -1.0 	 Total Reward: -1.0
Reward: -1.0 	 Total Reward: -1.0
Reward: -1.0 	 Total Reward: -1.0
Reward: -58.0 	 Total Reward: -59.0
Reward: -67.5 	 Total Reward: -68.5
Reward: -58.0 	 Total Reward: -59.0
Reward: -58.0 	 Total Reward: -59.0
Reward: -73.0 	 Total Reward: -132.0
Reward: -54.0 	 Total Reward: -122.5
Reward: -73.0 	 Total Reward: -132.0
Reward: -101.5 	 Total Reward: -160.5
Reward: -67.5 	 Total Reward: -199.5
Reward: -86.5 	 Total Reward: -209.0
Reward: -77.0 	 Total Reward: -209.0
Reward: -86.5 	 Total Reward: -247.0
Reward: -73.5 	 Total Reward: -273.0
Reward: -64.0 	 Total Reward: -273.0
Reward: -92.5 	 Total Reward: -301.5
Reward: -64.0 	 Total Reward: -311.0
Reward: -39.5 	 Total Reward: -312.5
Reward: -30.0 	 Total Reward: -303.0
Reward: -30.0 	 Total Reward: -331.5
Reward: -49.0 	 Total Reward: -360.0
Reward: -35.0 	 Total Reward: -347.5
Reward: -25.5 	 Total Reward: -328.5
Reward: -16.0 	 Total

Epoch #1:   0%|          | 0/10000 [00:00<?, ?it/s]

Reward: -1.0 	 Total Reward: -1.0
Reward: -10.5 	 Total Reward: -10.5
Reward: -1.0 	 Total Reward: -1.0
Reward: -1.0 	 Total Reward: -1.0
Reward: -34.0 	 Total Reward: -35.0
Reward: 3.5 	 Total Reward: -7.0
Reward: -31.0 	 Total Reward: -32.0
Reward: -73.0 	 Total Reward: -74.0
Reward: -54.0 	 Total Reward: -89.0
Reward: -76.5 	 Total Reward: -83.5
Reward: -43.0 	 Total Reward: -75.0
Reward: -120.0 	 Total Reward: -194.0
Reward: 33.5 	 Total Reward: -55.5
Reward: -134.5 	 Total Reward: -218.0
Reward: -41.5 	 Total Reward: -116.5
Reward: -105.5 	 Total Reward: -299.5
Reward: -9.0 	 Total Reward: -64.5
Reward: -43.0 	 Total Reward: -261.0
Reward: -20.5 	 Total Reward: -137.0
Reward: -71.5 	 Total Reward: -371.0
Reward: -1.5 	 Total Reward: -66.0
Reward: -106.5 	 Total Reward: -367.5
Reward: -30.0 	 Total Reward: -167.0
Reward: -50.0 	 Total Reward: -421.0
Reward: -14.5 	 Total Reward: -80.5
Reward: -27.0 	 Total Reward: -394.5
Reward: -11.5 	 Total Reward: -178.5
Reward: -46.0 	 Total Re

Epoch #1:  10%|#         | 1000/10000 [04:46<42:55,  3.49it/s, env_step=1000, gradient_step=100, len=0, n/ep=0, n/st=1000, rew=0.00]

Reward: -44.0 	 Total Reward: -124.5
Reward: -22.0 	 Total Reward: -416.5
Reward: 5.5 	 Total Reward: -173.0
Reward: -11.5 	 Total Reward: -478.5
Reward: 7.5 	 Total Reward: -117.0
Reward: -20.0 	 Total Reward: -436.5
Reward: -12.5 	 Total Reward: -185.5
Reward: -11.5 	 Total Reward: -490.0
Reward: -1.0 	 Total Reward: -118.0
Reward: 0.0 	 Total Reward: -436.5
Reward: -2.0 	 Total Reward: -187.5
Reward: -9.5 	 Total Reward: -499.5
Reward: -1.0 	 Total Reward: -119.0
Reward: -1.0 	 Total Reward: -437.5
Reward: 0.0 	 Total Reward: -187.5
Reward: -1.0 	 Total Reward: -500.5
Reward: 9.5 	 Total Reward: -109.5
Reward: -1.0 	 Total Reward: -438.5
Reward: -1.0 	 Total Reward: -188.5
Reward: 1.0 	 Total Reward: -499.5
Reward: -10.5 	 Total Reward: -120.0
Reward: 9.5 	 Total Reward: -429.0
Reward: -1.0 	 Total Reward: -189.5
Reward: -10.5 	 Total Reward: -510.0
Reward: -10.5 	 Total Reward: -130.5
Reward: -1.0 	 Total Reward: -430.0
Reward: -1.0 	 Total Reward: -190.5
Reward: -1.0 	 Total Rewar

...

Reward: -26.0 	 Total Reward: -1795.0
Reward: -37.0 	 Total Reward: -1987.0
Reward: -95.0 	 Total Reward: -1757.5
Reward: -143.0 	 Total Reward: -3333.5
Reward: -41.5 	 Total Reward: -1836.5
Reward: -47.0 	 Total Reward: -2034.0
Reward: 24.0 	 Total Reward: -1733.5
Reward: -3.5 	 Total Reward: -3337.0
Reward: -27.0 	 Total Reward: -1863.5
Reward: -11.5 	 Total Reward: -2045.5
Reward: -56.5 	 Total Reward: -1790.0
Reward: 17.0 	 Total Reward: -3320.0
Reward: -24.0 	 Total Reward: -1887.5
Reward: -4.0 	 Total Reward: -2049.5
Reward: 27.5 	 Total Reward: -1762.5
Reward: 5.5 	 Total Reward: -3314.5
Reward: -2.0 	 Total Reward: -1889.5
Reward: -12.5 	 Total Reward: -2062.0
Reward: -12.5 	 Total Reward: -1775.0
Reward: -12.5 	 Total Reward: -3327.0
Reward: -1.0 	 Total Reward: -1890.5
Reward: -11.5 	 Total Reward: -2073.5
Reward: 1.0 	 Total Reward: -1774.0
Reward: 8.5 	 Total Reward: -3318.5
Reward: -10.5 	 Total Reward: -1901.0
Reward: -1.0 	 Total Reward: -2074.5
Reward: 0.0 	 Total Rewar

Epoch #1:  90%|######### | 9000/10000 [54:05<06:09,  2.71it/s, env_step=9000, gradient_step=900, len=368, n/ep=4, n/st=1000, rew=-415.38]

Reward: -73.5 	 Total Reward: -2140.5
Reward: -116.0 	 Total Reward: -2410.0
Reward: -146.0 	 Total Reward: -1922.0
Reward: -38.0 	 Total Reward: -3653.0
Reward: -27.0 	 Total Reward: -2167.5
Reward: 47.5 	 Total Reward: -2362.5
Reward: -122.0 	 Total Reward: -2044.0
Reward: -141.0 	 Total Reward: -3794.0
Reward: 25.5 	 Total Reward: -2142.0
Reward: 29.5 	 Total Reward: -2333.0
Reward: -83.0 	 Total Reward: -2127.0
Reward: -61.0 	 Total Reward: -3855.0
Reward: 26.5 	 Total Reward: -2115.5
Reward: 60.5 	 Total Reward: -2272.5
Reward: -24.5 	 Total Reward: -2151.5
Reward: -16.0 	 Total Reward: -3871.0
Reward: 29.0 	 Total Reward: -2086.5
Reward: 50.0 	 Total Reward: -2222.5
Reward: -7.0 	 Total Reward: -2158.5
Reward: -15.5 	 Total Reward: -3886.5
Reward: -3.0 	 Total Reward: -2089.5
Reward: -4.0 	 Total Reward: -2226.5
Reward: -23.0 	 Total Reward: -2181.5
Reward: 0.0 	 Total Reward: -3886.5
Reward: 18.0 	 Total Reward: -2071.5
Reward: 16.0 	 Total Reward: -2210.5
Reward: -10.5 	 Total 

Epoch #1: 10001it [59:09,  2.82it/s, env_step=10000, gradient_step=1000, len=368, n/ep=0, n/st=1000, rew=-415.38]                           


Reward: -10.5 	 Total Reward: -955.5
Reward: -10.5 	 Total Reward: -908.0
Reward: 9.5 	 Total Reward: -935.5
Reward: 9.5 	 Total Reward: -1011.5
Reward: 170.5 	 Total Reward: -785.0
Reward: 90.5 	 Total Reward: -817.5
Reward: 120.0 	 Total Reward: -815.5
Reward: 110.5 	 Total Reward: -901.0
Reward: 313.0 	 Total Reward: -472.0
Reward: 130.0 	 Total Reward: -687.5
Reward: 196.5 	 Total Reward: -619.0
Reward: 293.0 	 Total Reward: -608.0
Reward: 217.5 	 Total Reward: -254.5
Reward: 82.0 	 Total Reward: -605.5
Reward: 69.5 	 Total Reward: -549.5
Reward: 311.0 	 Total Reward: -297.0
Reward: 98.5 	 Total Reward: -156.0
Reward: -66.0 	 Total Reward: -671.5
Reward: 169.0 	 Total Reward: -380.5
Reward: 245.5 	 Total Reward: -51.5
Reward: 140.0 	 Total Reward: -16.0
Reward: -2.0 	 Total Reward: -673.5
Reward: 92.5 	 Total Reward: -288.0
Reward: 192.5 	 Total Reward: 141.0
Reward: 69.0 	 Total Reward: 53.0
Reward: 20.5 	 Total Reward: -653.0
Reward: 20.5 	 Total Reward: -267.5
Reward: 12.0 	 Tot